![DSME-logo](./static/DSME_logo.png)

#  Reinforcement Learning and Learning-based Control

<p style="font-size:12pt";> 
<b> Prof. Dr. Sebastian Trimpe, Dr. Friedrich Solowjow </b><br>
<b> Institute for Data Science in Mechanical Engineering (DSME) </b><br>
<a href = "mailto:rllbc@dsme.rwth-aachen.de">rllbc@dsme.rwth-aachen.de</a><br>
</p>

---
Reinforce Implementation

Notebook Authors: Ramil Sabirov

## Library Imports

In [ ]:
# Run this cell in Google Colab before importing the notebook dependencies.
%pip install -q \
    "gymnasium[classic-control]==0.29.1" \
    "easydict==1.13" \
    "wandb==0.20.1" \
    "ruamel.yaml==0.18.14" \
    "tensorboard==2.19.0" \
    "pandas==2.3.0" \
    "matplotlib==3.10.3" \
    "tqdm==4.67.1"

In [ ]:
import os
import time
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from tqdm import notebook
from ruamel.yaml import YAML
from easydict import EasyDict as edict
from IPython.display import Video

import gymnasium as gym
import wandb
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter

warnings.filterwarnings("ignore", category=DeprecationWarning)

os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['WANDB_NOTEBOOK_NAME'] = 'reinforce_baseline_student.ipynb'

plt.rcParams['figure.dpi'] = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def make_single_env(env_id, seed):
    env = gym.make(env_id)
    env = gym.wrappers.RecordEpisodeStatistics(env)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    return env


def make_env(env_id, seed):
    def thunk():
        env = gym.make(env_id)
        env = gym.wrappers.RecordEpisodeStatistics(env)
        env.reset(seed=seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)
        return env
    return thunk


def make_log_dir():
    log_dir = os.path.abspath('logs')
    if not os.path.isdir(log_dir):
        os.mkdir(log_dir)
    return log_dir


def wandb_logging(wandb_prj_name, run_name, config=None, save_code=False):
    log_dir = make_log_dir()
    wandb.login()
    wandb.init(
        dir=log_dir,
        project=wandb_prj_name,
        sync_tensorboard=True,
        name=run_name,
        config=config,
        save_code=save_code,
    )


def setup_logging(wandb_prj_name, exp_dict, hypp_dict):
    if wandb.run is not None:
        wandb.finish()
    if exp_dict.enable_wandb_logging:
        wandb_logging(wandb_prj_name, exp_dict.run_name, dict(exp_dict, **hypp_dict))
    else:
        wandb.init(mode="disabled")
    exp_folder = "" if exp_dict.exp_type is None else exp_dict.exp_type
    tb_writer = SummaryWriter(f"logs/{exp_folder}/{exp_dict.run_name}/tb")
    tb_writer.add_text(
        "hyperparameters",
        "|param|value|\n|-|-|\n%s" % "\n".join(
            [f"|{key}|{value}|" for key, value in dict(exp_dict, **hypp_dict).items()]
        ),
    )
    return tb_writer


def create_folder_relative(folder_name, assert_flag=False):
    log_dir = make_log_dir()
    abs_folder_path = os.path.abspath(f"{log_dir}/{folder_name}")
    if not os.path.isdir(abs_folder_path):
        if assert_flag:
            raise AssertionError(f"Following folder does not exist {abs_folder_path}")
        os.makedirs(abs_folder_path)
        folder_already_exist = False
    else:
        folder_already_exist = True
    return abs_folder_path, folder_already_exist


def save_train_config_to_yaml(exp_dict, hypparam_dict):
    exp_folder = "" if exp_dict.exp_type is None else exp_dict.exp_type
    folder_path, _ = create_folder_relative(f"{exp_folder}/{exp_dict.run_name}")
    yaml = YAML()
    yaml.default_flow_style = False
    file_full_path = f"{folder_path}/experiment_config.yml"
    with open(file_full_path, 'w') as f:
        yaml.dump(
            dict(experiment_parameters=dict(exp_dict), hyperparameters=dict(hypparam_dict)),
            f,
        )


def save_tracked_values(returns_over_runs, episode_len_over_runs, episode_list, eval_count, run_name, exp_type=None):
    exp_folder = "" if exp_type is None else exp_type

    eval_id = [f"eval_idx{i:02d}" for i in range(eval_count)]
    sub_run_index = np.repeat([eval_id], len(returns_over_runs), axis=0).reshape(-1)

    episode_list = np.repeat(episode_list, eval_count, axis=0)
    returns_over_runs = np.array(returns_over_runs).reshape(-1)
    episode_len_over_runs = np.array(episode_len_over_runs).reshape(-1)

    df = pd.DataFrame(
        data=[episode_list[:, 0], episode_list[:, 1], sub_run_index, returns_over_runs, episode_len_over_runs]
    ).T
    df.columns = ['episode', 'global_step', 'sub_run_index', 'returns', 'episode_length']

    folder_path, _ = create_folder_relative(f"{exp_folder}/{run_name}")
    csv_full_path = f"{folder_path}/tracked_performance_training.csv"
    with open(csv_full_path, 'wb') as f:
        df.to_csv(f, index=False)


def save_model(model, run_name, exp_type=None, print_path=True):
    exp_folder = "" if exp_type is None else exp_type
    folder_path, _ = create_folder_relative(f"{exp_folder}/{run_name}")
    model_full_path = f"{folder_path}/agent_model.pt"
    with open(model_full_path, 'wb') as f:
        torch.save(model, f)
    if print_path:
        print(f"Agent model saved to path: \n{model_full_path}")


def load_model(run_name=None, folder_path=None, exp_type=None):
    exp_folder = "" if exp_type is None else exp_type
    if run_name is None:
        raise ValueError("input run_name missing")
    if folder_path is None:
        folder_path, _ = create_folder_relative(f"{exp_folder}/{run_name}", assert_flag=True)
    model_full_path = f"{folder_path}/agent_model.pt"
    return torch.load(
        model_full_path,
        map_location=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
        weights_only=False,
    )


def evaluate_agent(envs, model, run_count, seed, greedy_actor=False):
    next_obs, _ = envs.reset(seed=list(range(seed, seed + envs.num_envs)))
    next_obs = torch.Tensor(next_obs).to(device)
    returns_over_runs = []
    episode_len_over_runs = []
    finish = False
    model.eval()
    while not finish:
        with torch.no_grad():
            actions = model.get_action(next_obs, greedy_actor)
        next_obs, rewards, terminated, truncated, info = envs.step(actions.cpu().numpy())
        next_obs = torch.Tensor(next_obs).to(device)
        if "final_info" in info.keys():
            for final_info_single in info["final_info"]:
                if final_info_single is not None and "episode" in final_info_single.keys():
                    returns_over_runs.append(final_info_single["episode"]["r"])
                    episode_len_over_runs.append(final_info_single["episode"]["l"])
                    if len(returns_over_runs) >= run_count:
                        finish = True
                        break
    model.train()
    return returns_over_runs, episode_len_over_runs


def record_video(env_id, agent, file, exp_type=None, greedy=False, env_wrapper=None):
    frames = []
    env_wrapper = [] if env_wrapper is None else env_wrapper
    env = gym.make(env_id, render_mode="rgb_array")
    for wrapper in env_wrapper:
        env = wrapper(env)

    if isinstance(agent, str):
        agent = load_model(run_name=agent, exp_type=exp_type)

    state, _ = env.reset()
    done = False
    while not done:
        with torch.no_grad():
            action = agent.get_action(torch.Tensor(state).unsqueeze(0).to(device), greedy=greedy)

        state, _, terminated, truncated, info = env.step(action.squeeze(0).cpu().numpy())
        done = terminated or truncated
        frames.append(env.render())

    env.close()

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis('off')
    img = plt.imshow(frames[0])

    def animate(frame):
        img.set_data(frames[frame])
        return [img]

    anim = FuncAnimation(fig, animate, frames=len(frames), interval=20)
    plt.close()
    anim.save(file, writer="ffmpeg", fps=30)


def save_and_log_agent(exp_dict, agent, episode_step, greedy=False, print_path=True, env_wrapper=None):
    env_wrapper = [] if env_wrapper is None else env_wrapper
    save_model(agent, exp_dict.run_name, exp_type=exp_dict.exp_type, print_path=print_path)
    exp_folder = "" if exp_dict.exp_type is None else exp_dict.exp_type
    if exp_dict.capture_video:
        filepath, _ = create_folder_relative(f"{exp_folder}/{exp_dict.run_name}/videos")
        video_file = f"{filepath}/{episode_step}.mp4"
        record_video(exp_dict.env_id, agent, video_file, greedy=greedy, env_wrapper=env_wrapper)
        if wandb.run is not None:
            wandb.log({"video": wandb.Video(video_file, format="mp4")})

## Initializations

### Experiment Init

We primarily use dictionaries for initializing experiment parameters and training hyperparameters. We use the `EasyDict` (imported as `edict`) library, which allows us to access dict values as attributes while retaining the operations and properties of the original python `dict`! [[Github Link](https://github.com/makinacorpus/easydict)]

In this notebook we use a few `edicts` with `exp` being one of them. It is initialized in the following cell and has keys and values containing information about the experiment. Although the dict is initialized in this section, we keep adding new keys and values to the dict in the later sections as well.  

This notebook supports gym environments with observation space of type `gym.spaces.Box` and action space of type `gym.spaces.Discrete`. Eg: Acrobot-v1, CartPole-v1, MountainCar-v0

In [7]:
exp = edict()

exp.exp_name = 'REINFORCE'  # algorithm name, in this case it should be 'REINFORCE'
exp.env_id = 'CartPole-v1'  # name of the gym environment to be used in this experiment. Eg: Acrobot-v1, CartPole-v1, MountainCar-v0
exp.device = device.type  # save the device type used to load tensors and perform tensor operations

set_random_seed = True  # set random seed for reproducibility of python, numpy and torch
exp.seed = int(os.getenv("SEED", 1))

# name of the project in Weights & Biases (wandb) to which logs are patched. (only if wandb logging is enabled)
# if the project does not exist in wandb, it will be created automatically
wandb_prj_name = f"RLLBC_{exp.env_id}"

# name prefix of output files generated by the notebook
exp.run_name = f"{exp.env_id}__{exp.exp_name}__{exp.seed}__{datetime.now().strftime('%y%m%d_%H%M%S')}"

if set_random_seed:
    random.seed(exp.seed)
    np.random.seed(exp.seed)
    torch.manual_seed(exp.seed)
    torch.backends.cudnn.deterministic = set_random_seed

### Agent Model Class

The `Agent` class consists of a deep MLP policy that is trained during training. The network takes as input the representation of the state, passes it through several hidden layers, and finally evaluates to a probability distribution over all actions with the `softmax` function.

The `Agent` class has two methods:
1. `get_action_logprob_and_value` evaluates the network and samples an action from the resulting probability distribution. It also returns the logarithm of the action probability $\log \pi(a_t| s_t)$ which is used for obtaining gradient estimates for training. In addition, it returns the current value estimate $\hat{V}(s_t)$ of the state.

2. `get_action` evaluates the network to the probability distribution and either samples an action from that distribution (greedy = false) or returns the action with the maximal probability (greedy = true).

In [8]:
class Agent(nn.Module):
    def __init__(self, env):
        super().__init__()
        self.p_network = nn.Sequential(
            nn.Linear(np.array(env.observation_space.shape).prod(), 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, env.action_space.n),
            nn.Softmax(dim=-1)
        )
        self.v_network = nn.Sequential(
            nn.Linear(np.array(env.observation_space.shape).prod(), 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

    def get_action_logprob_and_value(self, x):

        action_probs = self.p_network(x)
        probs = Categorical(probs=action_probs)
        action = probs.sample()

        return action, probs.log_prob(action), self.v_network(x)

    def get_action(self, x, greedy=False):
        action_probs = self.p_network(x)

        if greedy:
            action = action_probs.argmax(dim=1)
        else:
            probs = Categorical(probs=action_probs)
            action = probs.sample()

        return action

### Training Params & Agent Hyperparams

The parameters and hyperparameters in this section are broadly categorized as below:
1. Flags for logging: 
    - Stored in the `exp` dict. 
    - This notebook uses tensorboard logging by default to log experiment metrics. These tb log files are saved in the directory `logs/<exp.exp_type>/<exp.run_name>/tb`. (to learn about `exp.exp_type` refer point 3. below)
    - To enable logging of gym videos of the agent's interaction with the env set `exp.capture_video = True`
    - Patch tensorboard logs and gym videos to Weigths & Biases (wandb) by setting `exp.enable_wandb_logging = True`
2. Flags and parameters to generate average performance throughout training:
    - Stored in the `exp` dict
    - If `exp.eval_agent = True`, the performance of the agent during it's training is saved in the corresponding logs folder. You can later used this to compare the performance of your current agent with other agents during their training (in Section 1.4.2).
    - Every `exp.eval_frequency` episodes the trained agent is evaluated using the `envs_eval` by playing out `exp.eval_count` episodes
    - To speed up training set `exp.eval_agent = False` 
3. Create experiment hierarchy inside log folders:
    - if `exp.exp_type` is None, experiment logs are saved to the root log directory `logs`, ie, `/logs/<exp.run_name>`, otherwise they are saved to the directory `logs/<exp.exp_type>/<exp._name>`
4. Parameters and hyperparameters related to the algorithm:
    - Stored in the `hypp` dict
    - Quick reminder:  the `num_steps` key in the `hypp` dict is also a hyperparameter defined in Env & Rollout Buffer Init Section.

Note: 
1. If Weigths and Biases (wandb) logging is enabled, when you run the "Training The Agent" cell, enter your wandb's api key when prompted. 
2. Training takes longer when either gym video recording or agent evaluation during training is enabled. To speed up training set both `exp.capture_video` and `exp.eval_agent` to `False`.

In [9]:
hypp = edict()

# flags for logging purposes
exp.enable_wandb_logging = False
exp.capture_video = bool(os.getenv("CAPTURE_VIDEO", True))

# flags to generate agent's average performance during training
exp.eval_agent = bool(os.getenv("EVAL", True))  # disable to speed up training
exp.eval_count = 10
exp.eval_frequency = 20

# putting the run into the designated log folder for structuring
exp.exp_type = None  # directory the run is saved to. Should be None or a string value

# agent training specific parameters and hyperparameters
hypp.total_timesteps = 300_000  # the training duration in number of time steps
hypp.learning_rate = 3e-4  # the learning rate for the optimizer
hypp.gamma = 0.99  # decay factor of future rewards

## Training the Agent

Before we begin training the agent, we first initialize the logging (based on the respective flags in the `exp` dict), the object of the `Agent` class, and the optimizer, followed by an initial set of observations. 


After that comes the main training loop which is comprised of:  
1. Collect the trajectory i.e. executing the environment until it is finished, while keeping track of received rewards
2. Compute the returns $G_t$ for every step $t$ of the trajectory
3. Compute the policy loss with baseline $L_p = -\sum_{t=1}^{n}\log \pi_\theta(a_t|s_t) * (G_t - \hat{V}(s_t))$.
4. Compute the value loss $L_v = \frac{1}{n}\sum_{t=1}^{n}(G_t-\hat{V}(s_t))^2$
5. Perform gradient descent (which is equivalent to gradient ascent regarding the gradient $\frac{\delta}{\delta\theta}{-L}$)

Post completion of the main training loop, we save a copy of the following in the directory `logs/<exp.exp_type>/<exp.run_name>`:
1. `exp` and `hypp` dicts into a `.config` file 
2. `agent` (instance of `Agent` class) into a `.pt` file for later evaluation
3. agent performance progress throughout training into a `.csv` file if `exp.eval_agent=True`


Note: we have two gym environments, `envs` and `envs_eval` in the initalizations. `envs` is used to fill the rollout buffer with trajectories and `envs_eval` is used to evaluate the agent performance at different stages of training.

In [10]:
# Init tensorboard logging and wandb logging
writer = setup_logging(wandb_prj_name, exp, hypp)

env = make_single_env(exp.env_id, exp.seed)
envs_eval = gym.vector.SyncVectorEnv([make_env(exp.env_id, exp.seed + i) for i in range(1)])

# init list to track agent's performance throughout training
tracked_returns_over_training = []
tracked_episode_len_over_training = []
tracked_episode_count = []
greedy_evaluation = False
eval_max_return = -float('inf')

agent = Agent(env).to(device)
optimizer = optim.Adam(agent.parameters(), lr=hypp.learning_rate)

start_time = time.time()

pbar = notebook.tqdm(total=hypp.total_timesteps)

# the maximum number of steps an evironment is rolled out
max_steps = 1000

global_step = 0
episode_step = 0
gradient_step = 0

while global_step < hypp.total_timesteps:

    next_obs, _ = env.reset()
    rewards = torch.zeros(max_steps).to(device)
    actions = torch.zeros((max_steps,) + env.action_space.shape).to(device)
    obs = torch.zeros((max_steps, ) + env.observation_space.shape).to(device)
    logprobs = torch.zeros(max_steps).to(device)
    values = torch.zeros(max_steps).to(device)

    episode_length = 0

    # collect trajectory
    for step in range(max_steps):

        episode_length = episode_length + 1
        global_step = global_step + 1

        next_obs = torch.tensor(next_obs).to(device)
        obs[step] = next_obs

        # choose action according to agent network
        action, log_prob, value = agent.get_action_logprob_and_value(next_obs)

        # apply action to envs
        next_obs, reward, terminated, truncated, info = env.step(action.cpu().item())
        done = terminated or truncated

        rewards[step] = torch.tensor(reward).to(device)
        actions[step] = action
        logprobs[step] = log_prob
        values[step] = value

        if done:
            # episode has been finished
            episode_step = episode_step + 1
            break

    # evaluate model
    if (episode_step % exp.eval_frequency == 0) and exp.eval_agent:
        tracked_return, tracked_episode_len = evaluate_agent(envs_eval, agent, exp.eval_count,
                                                                exp.seed, greedy_actor=greedy_evaluation)
        tracked_returns_over_training.append(tracked_return)
        tracked_episode_len_over_training.append(tracked_episode_len)
        tracked_episode_count.append([episode_step, global_step])

        # if there has been improvement of the model
        if np.mean(tracked_return) > eval_max_return:
            eval_max_return = np.mean(tracked_return)
            # call helper function - save model, create video, log video to wandb
            save_and_log_agent(exp, agent, episode_step, greedy=greedy_evaluation, print_path=False)

    # calculate returns
    returns = torch.zeros(episode_length, device=device)
    for t in reversed(range(episode_length)):

        if t == episode_length-1:
            returns[t] = rewards[t]
        else:
            returns[t] = returns[t+1] * hypp.gamma + rewards[t]

    # calculate loss from tensors
    policy_loss = torch.zeros((1,), device=device)
    for t in range(episode_length):
        policy_loss -= returns[t] * logprobs[t]
        #SOLUTION GOES HERE

    #SOLUTION GOES HERE

    loss = policy_loss + value_loss

    # logging information regarding agent performance
    writer.add_scalar("rollout/episodic_return", sum(rewards), global_step)
    writer.add_scalar("rollout/episodic_length", episode_length, global_step)

    # logging information about the loss
    writer.add_scalar("train/policy_loss", policy_loss, global_step)
    writer.add_scalar("train/value_loss", value_loss, global_step)
    writer.add_scalar("others/SPS", int(global_step / (time.time() - start_time)), global_step)
    writer.add_scalar("Charts/gradient_step", gradient_step, global_step)
    writer.add_scalar("Charts/episode_step", episode_step, global_step)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    gradient_step = gradient_step + 1
    pbar.update(min(episode_length, hypp.total_timesteps - pbar.n))
    pbar.set_description(f"episode={episode_step}, episodic_return={sum(rewards)}")

# one last evaluation stage
if exp.eval_agent:
    tracked_return, tracked_episode_len = evaluate_agent(envs_eval, agent, exp.eval_count,
                                                            exp.seed, greedy_actor=greedy_evaluation)
    tracked_returns_over_training.append(tracked_return)
    tracked_episode_len_over_training.append(tracked_episode_len)
    tracked_episode_count.append([episode_step, global_step])

    # if there has been improvement of the model - save model, create video, log video to wandb
    if np.mean(tracked_return) > eval_max_return:
        eval_max_return = np.mean(tracked_return)
        # call helper function - save model, create video, log video to wandb
        save_and_log_agent(exp, agent, episode_step, greedy=greedy_evaluation, print_path=True)

    save_tracked_values(tracked_returns_over_training, tracked_episode_len_over_training,
                           tracked_episode_count, exp.eval_count, exp.run_name, exp.exp_type)    

env.close()
writer.close()
pbar.close()
if wandb.run is not None:
    wandb.finish(quiet=True)
    wandb.init(mode= 'disabled')

save_train_config_to_yaml(exp, hypp)

Charts/episode_step,▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇▇▇██
Charts/gradient_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇███
global_step,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇█
others/SPS,▁▃▃▂▄▄▄▄▅▅▆▆▅▅▅▅▅▅▆▅▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇███▇
rollout/episodic_length,▁▁▁▁▁▂▁▁▂▂▂▁▂▂▃▃▂▃▃▄▃▃▆█▄▃▃▂▃▃▃▃▃▃▃▃▃▃▃▅
rollout/episodic_return,▁▁▁▁▂▁▁▁▂▁▂▂▂▂▃▃▃▄▆▇▃▃▃▃▃▃▃▃▃▃▃▃▃▃▂▄▅███
train/policy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▁▂▂▂▂▃▄▂█▅▁▁▁▁▁▁▁▁▁▁▁▁▂▆
train/value_loss,▁▁▁▂▁▁▂▃▁▅▂▂▂▁▁▄▄▅▅▃▄▄██▇▃▃▂▂▂▂▂▂▂▂▂▂▆▅▅
Charts/episode_step,185
Charts/gradient_step,184
global_step,29983


wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`


  0%|          | 0/300000 [00:00<?, ?it/s]

wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING `fps` argument does not affect the frame rate of the video when providing a file path or raw bytes.
wandb: WARNING The `quiet` argument to `wandb.run.finish()` is deprecated, use `wandb.Settings(quiet=...)` to set this instead.


Charts/episode_step,▁▁▂▂▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
Charts/gradient_step,▁▁▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇▇██
global_step,▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▆▆▆█
others/SPS,▃▂▃▁▂▁▁▁▁▁▁▁▁▁▁▂▂▂▂▃▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇█████
rollout/episodic_length,▂▅▁▁▁▁▁▁▁▂▃▃▃▄▄▅▆▇██▅▂▂▂▃▅▆▇████████████
rollout/episodic_return,▃▂▃▅▅▁▁▁▁▁▁▁▁▃▃▃██▂▂▄▄▄▆▆▇▇█████████████
train/policy_loss,▂▂▂▅▅█▂▂▂▂▂▂▂▂▂▂▂▃▂▆▂▂▂▂▂▂▂▃▂▃▂▂▁▂▁▂▂▂▂▂
train/value_loss,█▇▆▁▁▁▁▁▁▁▂▂▃▄▅▇▇▆▅▂▂▁▁▁▁▂▂▄▂▅▅▅▅▅▅▅▅▅▅▅
Charts/episode_step,1304
Charts/gradient_step,1303
global_step,300151


In [8]:
# %load_ext tensorboard
# %tensorboard --logdir logs --host localhost